# Statistical analysis — expanded hypothesis family
**Calibration vs Deep Ensembles, MAKE revision**

This notebook runs the paired Wilcoxon family on the merged 7,740-row results CSV and
applies a single **global Holm-Bonferroni correction** across the full final family.

## Hypothesis groups

| Family | What it tests | # tests |
|---|---|---|
| H1 | Raw model comparisons (with CatBoost, MC-Dropout) | 18 |
| H2 | Calibrator effects within each main model | 60 |
| H3 | Post-calibration ensemble (M=5) vs comparison models | 30 |
| **H4 (new)** | Ensemble size effects: M=3 vs M=5 vs M=10 | 18 |
| **H5 (new)** | Member-level vs pseudo-logit TS within each ensemble | 9 |
| **H6 (new)** | MC-Dropout vs single MLP and vs deep ensemble | 12 |
| **H7 (new)** | Dirichlet ODIR vs MLR across all model families | 24 |
| **Total** | | **~171** |

## 1. Setup

In [ ]:
# Working directory for cached probs and intermediate CSVs.
# Override with:  export WORK_DIR=/path/to/persistent/storage
import os, pathlib

WORK_DIR = pathlib.Path(os.environ.get('WORK_DIR', './work')).resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)
PROBS_DIR = WORK_DIR / 'probs'
PROBS_DIR.mkdir(exist_ok=True)

print(f"Working directory: {WORK_DIR}")
print(f"Cached probability files: {len(list(PROBS_DIR.glob('*.npz')))}")


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(WORK_DIR / 'results_raw.csv')
print(f"Rows: {len(df)}")
print(f"Models: {sorted(df['model'].unique())}")
print(f"Calibrators: {sorted(df['calibrator'].unique())}")
print(f"Seeds: {sorted(df['seed'].unique())}")
print(f"Datasets: {df['task_id'].nunique()}")

# Sanity check
mat = df.groupby(['model', 'calibrator']).size().unstack(fill_value=0)
print(f"\nRow count matrix (target 180 per cell):")
print(mat.to_string())

## 2. Two-stage aggregation

Per the paper protocol: first take median across 5 seeds → one value per (task, model, calibrator). Then run paired tests on these per-dataset values, with n = 36 datasets.

In [ ]:
# Aggregate seeds → per-dataset median
per_ds = (df.groupby(['task_id', 'model', 'calibrator'])
          [['cal_nll', 'cal_ece_mean', 'cal_brier_score']]
          .median()
          .reset_index())
print(f"Per-dataset table: {len(per_ds)} rows (expected: {df['task_id'].nunique()} datasets × {df.groupby(['model', 'calibrator']).ngroups} model-cal cells)")
print(per_ds.head())

## 3. Helper: paired Wilcoxon + rank-biserial effect size

In [ ]:
from scipy.stats import wilcoxon

def paired_test(metric, model_a, cal_a, model_b, cal_b):
    """Paired Wilcoxon on per-dataset values. Returns dict or None if insufficient data."""
    a = per_ds[(per_ds.model == model_a) & (per_ds.calibrator == cal_a)].set_index('task_id')[metric]
    b = per_ds[(per_ds.model == model_b) & (per_ds.calibrator == cal_b)].set_index('task_id')[metric]
    a, b = a.align(b, join='inner')
    if len(a) < 5:
        return None
    
    diff = (a - b)
    
    # Handle case where all differences are zero (wilcoxon would fail)
    if (diff == 0).all():
        return {'n': len(a), 'med_a': a.median(), 'med_b': b.median(),
                'med_diff': 0.0, 'p': 1.0, 'rbc': 0.0}
    
    try:
        # zero_method='wilcox' is the default; suppress small-sample warnings
        stat, p = wilcoxon(a, b, zero_method='wilcox')
    except ValueError:
        return None
    
    # Rank-biserial correlation as effect size 
    n = len(a)
    pos = (diff > 0).sum()
    neg = (diff < 0).sum()
    total = pos + neg  # excludes zeros for the standard rbc
    rbc = (pos - neg) / max(total, 1) if total > 0 else 0.0
    # Sign convention: positive r means model_a > model_b (a worse if metric is loss-like)
    
    return {
        'n':        n,
        'med_a':    float(a.median()),
        'med_b':    float(b.median()),
        'med_diff': float(diff.median()),
        'p':        float(p),
        'rbc':      float(rbc),
    }


def magnitude(r):
    a = abs(r)
    if a < 0.3:   return 'small'
    if a < 0.5:   return 'medium'
    return 'large'

print("✓ paired_test helper defined")

## 4. Build the hypothesis family

In [ ]:
METRICS = ['cal_nll', 'cal_ece_mean', 'cal_brier_score']
tests = []

# ── H1: raw model comparisons (all under calibrator='none') ──
H1_pairs = [
    ('deep_ensemble', 'lgbm'),
    ('deep_ensemble', 'xgboost'),
    ('deep_ensemble', 'catboost'),      
    ('deep_ensemble', 'single_mlp'),
    ('mc_dropout',    'single_mlp'),     
    ('mc_dropout',    'deep_ensemble'),  
]
for a, b in H1_pairs:
    for m in METRICS:
        r = paired_test(m, a, 'none', b, 'none')
        if r:
            tests.append({'family': 'H1', 'comparison': f'{a} vs {b}',
                          'calibrator': 'none', 'metric': m,
                          'note': 'raw model comparison', **r})

# ── H2: calibrator effects within each model ──
H2_models = ['lgbm', 'xgboost', 'catboost', 'single_mlp', 'mc_dropout']
H2_cals   = ['temp', 'logistic', 'isotonic', 'dirichlet']
for model in H2_models:
    for cal in H2_cals:
        for m in METRICS:
            r = paired_test(m, model, cal, model, 'none')
            if r:
                tests.append({'family': 'H2', 'comparison': f'{cal} vs raw',
                              'model': model, 'calibrator': cal, 'metric': m,
                              'note': 'calibrator effect within model', **r})

# ── H3: post-calibration ensemble (M=5) vs comparison models ──
H3_pairs = [('deep_ensemble', m) for m in ['lgbm', 'xgboost', 'catboost', 'single_mlp', 'mc_dropout']]
for cal in ['temp', 'dirichlet']:
    for a, b in H3_pairs:
        for m in METRICS:
            r = paired_test(m, a, cal, b, cal)
            if r:
                tests.append({'family': 'H3', 'comparison': f'{a} vs {b}',
                              'calibrator': cal, 'metric': m,
                              'note': 'post-cal ensemble vs comparison', **r})

# ── H4: ensemble size effects ──
H4_pairs = [
    ('deep_ensemble_m3',  'deep_ensemble'),     # M=3  vs M=5
    ('deep_ensemble_m10', 'deep_ensemble'),     # M=10 vs M=5
    ('deep_ensemble_m10', 'deep_ensemble_m3'),  # M=10 vs M=3
]
for cal in ['temp', 'dirichlet']:
    for a, b in H4_pairs:
        for m in METRICS:
            r = paired_test(m, a, cal, b, cal)
            if r:
                tests.append({'family': 'H4', 'comparison': f'{a} vs {b}',
                              'calibrator': cal, 'metric': m,
                              'note': 'ensemble size effect', **r})

# ── H5: member-level vs pseudo-logit TS within each ensemble ──
for ens in ['deep_ensemble', 'deep_ensemble_m3', 'deep_ensemble_m10']:
    for m in METRICS:
        r = paired_test(m, ens, 'temp_member', ens, 'temp')
        if r:
            tests.append({'family': 'H5', 'comparison': 'temp_member vs temp',
                          'model': ens, 'metric': m,
                          'note': 'member-level vs pseudo-logit TS', **r})

# ── H6: MC-Dropout (BNN proxy) under temperature scaling + dirichlet ──
for cal in ['temp', 'dirichlet']:
    for a, b in [('mc_dropout', 'single_mlp'), ('mc_dropout', 'deep_ensemble')]:
        for m in METRICS:
            r = paired_test(m, a, cal, b, cal)
            if r:
                tests.append({'family': 'H6', 'comparison': f'{a} vs {b}',
                              'calibrator': cal, 'metric': m,
                              'note': 'BNN proxy comparison', **r})

# ── H7: Dirichlet ODIR vs MLR across all model families (new finding) ──
H7_models = ['lgbm', 'xgboost', 'catboost', 'single_mlp', 'mc_dropout',
             'deep_ensemble', 'deep_ensemble_m3', 'deep_ensemble_m10']
for model in H7_models:
    for m in METRICS:
        r = paired_test(m, model, 'dirichlet', model, 'logistic')
        if r:
            tests.append({'family': 'H7', 'comparison': 'dirichlet vs logistic',
                          'model': model, 'metric': m,
                          'note': 'Dirichlet ODIR replaces MLR', **r})

tests_df = pd.DataFrame(tests)
print(f"Total tests in family: {len(tests_df)}")
print(tests_df['family'].value_counts().sort_index().to_string())

## 5. Apply global Holm-Bonferroni correction

In [ ]:
from statsmodels.stats.multitest import multipletests

_, p_adj, _, _ = multipletests(tests_df['p'].values, alpha=0.05, method='holm')
tests_df['p_adj'] = p_adj
tests_df['sig']   = tests_df['p_adj'] < 0.05
tests_df['magnitude'] = tests_df['rbc'].apply(magnitude)

# Reorder columns for readability
front_cols = ['family', 'comparison', 'metric', 'calibrator', 'model',
              'n', 'med_a', 'med_b', 'med_diff', 'p', 'p_adj', 'sig', 'rbc', 'magnitude', 'note']
front_cols = [c for c in front_cols if c in tests_df.columns]
tests_df = tests_df[front_cols + [c for c in tests_df.columns if c not in front_cols]]

print(f"Total tests: {len(tests_df)}")
print(f"Significant after Holm correction: {tests_df['sig'].sum()}")
print(f"\nSignificant by family:")
print(tests_df.groupby('family')['sig'].agg(['sum', 'count']).rename(columns={'sum':'n_sig','count':'n_total'}).to_string())

## 6. Save results

In [ ]:
# Master CSV with everything
tests_df.to_csv(WORK_DIR / 'stats_master.csv', index=False)
print(f"✓ Saved: stats_master.csv ({len(tests_df)} rows)")

# Per-family tables
for fam in sorted(tests_df['family'].unique()):
    sub = tests_df[tests_df.family == fam]
    sub.to_csv(WORK_DIR / f'stats_{fam}.csv', index=False)
    print(f"  ✓ stats_{fam}.csv ({len(sub)} rows, {sub['sig'].sum()} significant)")

## 7. Headline findings

In [ ]:
def show_family(fam, title):
    print(f"\n{'='*70}")
    print(f"  {title}")
    print('='*70)
    sub = tests_df[tests_df.family == fam].copy()
    sub['mark'] = sub['sig'].apply(lambda s: ' *' if s else '  ')
    cols_to_show = [c for c in ['comparison','metric','calibrator','model','med_diff','p','p_adj','rbc','magnitude','mark']
                    if c in sub.columns]
    print(sub[cols_to_show].to_string(index=False,
                                       float_format=lambda x: f'{x:.4f}'))

show_family('H1', 'H1 — Raw model comparisons')
show_family('H2', 'H2 — Calibrator effects within each model')
show_family('H3', 'H3 — Post-calibration: ensemble (M=5) vs others')

In [ ]:
show_family('H4', 'H4 (new) — Ensemble size effects (M=3 vs M=5 vs M=10)')
show_family('H5', 'H5 (new) — Member-level vs pseudo-logit TS')
show_family('H6', 'H6 (new) — MC-Dropout (BNN proxy)')
show_family('H7', 'H7 (new) — Dirichlet ODIR vs MLR')